# Gemma 4 12B — Nepali ASR Prompt Development


This notebook performs **prompt development only** for the Nepali ASR benchmark. It uses the frozen prompt-development split and must not access the final 500-utterance benchmark during prompt selection.

Protocol:
- L0: fixed minimal control.
- L1: task-instruction candidates.
- L2: linguistically aware candidates.
- L3: selected L2 instruction + one fixed development-only demonstration.
- L4: selected L2 instruction + three fixed development-only demonstrations.
- Primary selection metric: corpus WER, after inference-failure rate.
- Secondary diagnostics: corpus CER, hallucination proxy, rejection/meta-response behavior, script presence, output length, and latency.


## A. Install dependencies


In [ ]:
import subprocess, sys

required = [
    "transformers==5.14.1",
    "accelerate>=1.6,<2",
    "librosa>=0.10,<1",
    "soundfile>=0.12,<1",
    "jiwer>=3.1,<5",
    "jsonlines>=4,<5",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *required]
)
print("Dependencies installed.")


## B. Configuration


In [ ]:
MODEL_ID = "google/gemma-4-12B-it"

import os, json, gc, random, hashlib, platform, re, unicodedata
from pathlib import Path
from datetime import datetime
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
import torch

MODEL_REV = "main"
EXPECTED_DEV_N = 30
SEED = 42

PRECISION = "auto_bf16_fp16"
QUANTIZATION = "none"

MAX_NEW_TOKENS = 256
DO_SAMPLE = False

PROMPT_DEV_ROOT = Path("/kaggle/input/datasets/alaxini/prompt-dev-1/stage1")
PROMPT_DEV_MANIFEST_PATH = PROMPT_DEV_ROOT / "metadata.csv"

RESULTS_ROOT = Path("/kaggle/working/results/prompt_dev/gemma4_12b")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB")


## C. Frozen evaluation normalization


In [ ]:
def normalize_for_eval(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFC", str(text)).lower()

    chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("P") or cat == "Nd":
            chars.append(" ")
        else:
            chars.append(ch)

    return re.sub(r"\s+", " ", "".join(chars)).strip()

assert normalize_for_eval("नेपाल, राम्रो छ।") == "नेपाल राम्रो छ"
assert normalize_for_eval("Test 123 टेस्ट १२३") == "test टेस्ट"

NORMALIZATION_RULES = {
    "unicode": "NFC",
    "latin_case": "lowercase",
    "remove_punctuation": True,
    "remove_decimal_digits": True,
    "preserve_devanagari": True,
    "preserve_latin": True,
    "collapse_whitespace": True,
}

print("Evaluation normalizer ready.")


## D. Load the prompt-development manifest


In [ ]:
if not PROMPT_DEV_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Metadata not found: {PROMPT_DEV_MANIFEST_PATH}")

raw_manifest = pd.read_csv(PROMPT_DEV_MANIFEST_PATH)

required_columns = {
    "file", "dataset_type", "original_file",
    "label_original", "label_normalized",
    "source", "duration", "duration_bin",
    "noise_type", "snr_db", "cmi", "cmi_band",
}

missing = required_columns - set(raw_manifest.columns)
if missing:
    raise ValueError(f"Missing metadata columns: {sorted(missing)}")

if len(raw_manifest) != EXPECTED_DEV_N:
    raise ValueError(
        f"Expected {EXPECTED_DEV_N} prompt-development utterances, "
        f"found {len(raw_manifest)}."
    )

manifest = raw_manifest.copy()
manifest["audio_path"] = manifest["file"].apply(
    lambda x: str(PROMPT_DEV_ROOT / str(x))
)

missing_audio = manifest[~manifest["audio_path"].map(os.path.exists)]
if len(missing_audio):
    display(missing_audio[["file", "audio_path"]].head(10))
    raise FileNotFoundError(
        f"{len(missing_audio)} prompt-development audio files are missing."
    )

manifest["utterance_id"] = manifest["file"].astype(str)
manifest["speech_condition"] = (
    manifest["dataset_type"].astype(str).str.strip().str.lower()
)
manifest["sample_id"] = (
    manifest["speech_condition"] + ":" + manifest["utterance_id"]
)

manifest["reference_original"] = (
    manifest["label_original"].fillna("").astype(str)
)
manifest["reference_dataset_normalized"] = (
    manifest["label_normalized"].fillna("").astype(str)
)
manifest["reference_normalized"] = (
    manifest["reference_dataset_normalized"].apply(normalize_for_eval)
)
manifest["duration_sec"] = pd.to_numeric(
    manifest["duration"], errors="coerce"
)

assert manifest["sample_id"].is_unique
assert manifest["audio_path"].map(os.path.exists).all()
assert manifest["reference_normalized"].str.len().gt(0).all()

conditions = set(manifest["speech_condition"].unique())
expected_conditions = {"clean", "noisy", "codeswitched"}
if conditions != expected_conditions:
    raise ValueError(
        f"Expected conditions {sorted(expected_conditions)}, "
        f"found {sorted(conditions)}."
    )

condition_counts = (
    manifest["speech_condition"].value_counts().sort_index().to_dict()
)
print("Condition counts:", condition_counts)

manifest_payload = (
    manifest[["sample_id", "speech_condition", "reference_normalized"]]
    .sort_values("sample_id")
    .to_json(orient="records", force_ascii=False)
)
MANIFEST_HASH = hashlib.sha256(
    manifest_payload.encode("utf-8")
).hexdigest()

print("Manifest SHA256:", MANIFEST_HASH)
display(manifest[[
    "sample_id", "speech_condition", "audio_path",
    "reference_original", "reference_normalized",
    "source", "duration_sec", "duration_bin",
    "noise_type", "snr_db", "cmi", "cmi_band",
]].head())


## E. Prompt candidates and fixed few-shot exemplars


In [ ]:
PROMPT_TEXTS = {
    "L0_a": (
        "Transcribe the following speech segment in its original language. "
        "Only output the transcription."
    ),
    "L1_a": (
        "You are a speech transcription system. "
        "Transcribe the following Nepali audio into Nepali text using Devanagari script. "
        "Produce a verbatim transcription. Do not translate. "
        "Return only the transcription, nothing else."
    ),
    "L1_b": (
        "Task: verbatim Nepali speech transcription.\n"
        "Language: Nepali (Devanagari script).\n"
        "Transcribe exactly what is spoken. Do not translate. "
        "Output only the transcription."
    ),
    "L1_c": (
        "Listen to the audio and write the exact Nepali words spoken in Devanagari. "
        "Do not translate, explain, or add anything. Return only the transcript."
    ),
    "L2_a": (
        "Transcribe the spoken Nepali audio verbatim in Devanagari script. "
        "Preserve any English words in Latin script and keep Nepali-English code-switching in the spoken order. "
        "Do not translate between languages. Keep fillers, repetitions, corrections, false starts, and incomplete words. "
        "Do not correct grammar or normalize the speaker's wording. Do not infer inaudible words. "
        "Do not add timestamps, speaker labels, explanations, or confidence scores. "
        "Return only the transcription."
    ),
    "L2_b": (
        "You are a verbatim transcription system for Nepali speech.\n"
        "Rules:\n"
        "1. Write Nepali in Devanagari.\n"
        "2. Write English words in Latin script.\n"
        "3. Preserve Nepali-English code-switching in the spoken order.\n"
        "4. Do not translate.\n"
        "5. Keep fillers, repetitions, corrections, false starts, and incomplete words.\n"
        "6. Do not correct grammar or normalize wording.\n"
        "7. Do not guess inaudible words.\n"
        "8. Add no timestamps, speaker labels, explanations, or confidence scores.\n"
        "9. Output only the transcription.\n"
        "10. Write spoken numbers as Nepali words, not digits."
    ),
    "L2_c": (
        "Write only the verbatim transcript of the audio. "
        "Use Devanagari for Nepali and Latin script for English; preserve code-switches exactly where spoken. "
        "Keep fillers, repetitions, corrections, false starts, and incomplete words. "
        "Do not translate, correct grammar, normalize wording, guess unclear speech, or add commentary."
    ),
}

PROMPT_META = {
    "L0_a": ("L0", "minimal_control"),
    "L1_a": ("L1", "plain_instruction"),
    "L1_b": ("L1", "structured_instruction"),
    "L1_c": ("L1", "compact_instruction"),
    "L2_a": ("L2", "linguistic_paragraph"),
    "L2_b": ("L2", "linguistic_rules"),
    "L2_c": ("L2", "linguistic_compact"),
}

def prompt_hash(text, shot_payload=None):
    payload = {"text": text, "shots": shot_payload or []}
    canonical = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def make_specs(prompt_ids):
    specs = {}
    for pid in prompt_ids:
        level, family = PROMPT_META[pid]
        text = PROMPT_TEXTS[pid]
        specs[pid] = {
            "prompt_id": pid,
            "level": level,
            "family": family,
            "text": text,
            "shot_count": 0,
            "prompt_hash": prompt_hash(text),
        }
    return specs

PROMPTSET_HASH = hashlib.sha256(
    json.dumps(PROMPT_TEXTS, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()

RUN_DIR = RESULTS_ROOT / (
    f"manifest_{MANIFEST_HASH[:12]}_prompts_{PROMPTSET_HASH[:12]}"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

with open(RUN_DIR / "prompt_candidates.json", "w", encoding="utf-8") as f:
    json.dump(PROMPT_TEXTS, f, indent=2, ensure_ascii=False)

manifest.to_csv(
    RUN_DIR / "prompt_dev_manifest_snapshot.csv",
    index=False,
)

MANUAL_SHOT_SAMPLE_IDS = {
    "clean": None,
    "codeswitched": None,
    "noisy": None,
}

def choose_representative(condition):
    subset = manifest[
        manifest["speech_condition"] == condition
    ].copy()
    if subset.empty:
        raise RuntimeError(f"No {condition} examples are available.")

    manual = MANUAL_SHOT_SAMPLE_IDS.get(condition)
    if manual:
        match = subset[subset["sample_id"] == manual]
        if len(match) != 1:
            raise ValueError(
                f"Manual shot ID not found uniquely: {manual}"
            )
        return match.iloc[0]

    median_duration = subset["duration_sec"].median()
    subset["duration_distance"] = (
        subset["duration_sec"] - median_duration
    ).abs()
    return subset.sort_values(
        ["duration_distance", "sample_id"]
    ).iloc[0]

def demonstration_transcript(row):
    original = str(
        row.get("reference_original", "") or ""
    ).strip()
    return original if original else row["reference_normalized"]

shot_rows = {
    c: choose_representative(c)
    for c in ["clean", "codeswitched", "noisy"]
}

SHOT_BANK = {}
for condition, row in shot_rows.items():
    SHOT_BANK[condition] = {
        "sample_id": row["sample_id"],
        "utterance_id": row["utterance_id"],
        "speech_condition": row["speech_condition"],
        "audio_path": row["audio_path"],
        "demonstration_transcript": demonstration_transcript(row),
        "duration_sec": float(row["duration_sec"]),
    }

shot_ids_all = {x["sample_id"] for x in SHOT_BANK.values()}
fewshot_eval_manifest = manifest[
    ~manifest["sample_id"].isin(shot_ids_all)
].copy()

assert len(fewshot_eval_manifest) == EXPECTED_DEV_N - 3
assert not set(fewshot_eval_manifest["sample_id"]) & shot_ids_all

with open(
    RUN_DIR / "fewshot_exemplars.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(SHOT_BANK, f, indent=2, ensure_ascii=False)

print("Prompt-set SHA256:", PROMPTSET_HASH)
print("Run directory:", RUN_DIR)
print("Fixed few-shot exemplars:")
for condition, ex in SHOT_BANK.items():
    print(
        condition,
        "|", ex["sample_id"],
        "|", f"{ex['duration_sec']:.2f}s"
    )
print("Few-shot evaluation targets:", len(fewshot_eval_manifest))


## F. Load the model

Gemma-specific implementation: the instruction precedes audio in each multimodal user turn. Deterministic generation is used throughout.


In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

print(f"Loading {MODEL_ID} @ {MODEL_REV} ...")

LOAD_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
    else torch.float16
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    padding_side="left",
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    dtype=LOAD_DTYPE,
    device_map="auto",
)
model.eval()

INPUT_DEVICE = next(model.parameters()).device

resolved_revision = (
    getattr(model.config, "_commit_hash", None)
    or getattr(
        getattr(processor, "tokenizer", None),
        "_commit_hash",
        None,
    )
    or MODEL_REV
)

def pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return "unknown"

environment = {
    "timestamp": datetime.now().isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "transformers": pkg_version("transformers"),
    "accelerate": pkg_version("accelerate"),

    "librosa": pkg_version("librosa"),
    "soundfile": pkg_version("soundfile"),
    "jiwer": pkg_version("jiwer"),
    "model_id": MODEL_ID,
    "requested_model_revision": MODEL_REV,
    "resolved_model_revision": resolved_revision,
    "precision_requested": PRECISION,
    "model_dtype": str(getattr(model, "dtype", "unknown")),
    "quantization": QUANTIZATION,
    "device_map": getattr(model, "hf_device_map", None),
    "input_device": str(INPUT_DEVICE),
    "manifest_sha256": MANIFEST_HASH,
    "promptset_sha256": PROMPTSET_HASH,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "normalization_rules": NORMALIZATION_RULES,
}

with open(
    RUN_DIR / "run_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        environment,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Resolved revision:", resolved_revision)
print("Model dtype:", getattr(model, "dtype", "unknown"))
print("Input device:", INPUT_DEVICE)
print("Device map:", getattr(model, "hf_device_map", "N/A"))


## G. Inference helpers and checkpointing


In [ ]:
def build_conversation(
    target_audio_path,
    prompt_text,
    exemplars=None,
):
    conversation = []

    for ex in (exemplars or []):
        conversation.append({
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Transcribe this speech segment verbatim. "
                        "Output only the transcription."
                    ),
                },
                {
                    "type": "audio",
                    "audio": ex["audio_path"],
                },
            ],
        })
        conversation.append({
            "role": "assistant",
            "content": ex["demonstration_transcript"],
        })

    conversation.append({
        "role": "user",
        "content": [
            {"type": "text", "text": prompt_text},
            {
                "type": "audio",
                "audio": target_audio_path,
            },
        ],
    })
    return conversation

def transcribe_one(
    audio_path,
    prompt_text,
    exemplars=None,
):
    conversation = build_conversation(
        audio_path,
        prompt_text,
        exemplars,
    )

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(INPUT_DEVICE, dtype=model.dtype)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[:, input_len:]

    decoded = processor.decode(
        new_tokens[0],
        skip_special_tokens=True,
    )
    return decoded, int(new_tokens.shape[1])



import time
import jsonlines
from tqdm.auto import tqdm

WRAPPER_PREFIXES = [
    r"^\s*\*{0,2}\s*transcription\s*\*{0,2}\s*:\s*",
    r"^\s*\*{0,2}\s*transcript\s*\*{0,2}\s*:\s*",
    r"^\s*\*{0,2}\s*output\s*\*{0,2}\s*:\s*",
    r"^\s*```(?:text)?\s*",
]

def clean_output(raw_output):
    text = (
        ""
        if raw_output is None
        else str(raw_output).strip()
    )
    for pattern in WRAPPER_PREFIXES:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE,
        )
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip(" \t\r\n`*")

def known_metadata_from_row(row):
    columns = [
        "dataset_type", "original_file", "source",
        "gender", "duration", "duration_sec",
        "duration_bin", "noise_type", "snr_db",
        "noise_file", "speakers", "cmi", "cmi_band",
    ]
    out = {}
    for c in columns:
        if c in row.index and pd.notna(row[c]):
            value = row[c]
            if isinstance(value, np.generic):
                value = value.item()
            out[c] = value
    return out

def run_pipeline(
    manifest_df,
    prompt_specs,
    output_path,
    exemplar_map=None,
):
    output_path = Path(output_path)
    exemplar_map = exemplar_map or {}

    completed = set()
    if output_path.exists():
        with jsonlines.open(output_path) as reader:
            for obj in reader:
                completed.add((
                    obj["sample_id"],
                    obj["prompt_hash"],
                    str(
                        obj.get(
                            "resolved_model_revision",
                            "",
                        )
                    ),
                ))

    valid_hashes = {
        spec["prompt_hash"]
        for spec in prompt_specs.values()
    }
    completed = {
        key
        for key in completed
        if key[1] in valid_hashes
        and key[2] == str(resolved_revision)
    }

    total = len(manifest_df) * len(prompt_specs)
    pbar = tqdm(
        total=total,
        initial=len(completed),
        desc=output_path.stem,
    )

    for _, row in manifest_df.iterrows():
        for pid, spec in prompt_specs.items():
            key = (
                row["sample_id"],
                spec["prompt_hash"],
                str(resolved_revision),
            )

            if key in completed:
                continue

            exemplars = exemplar_map.get(pid, [])

            record = {
                "model_id": MODEL_ID,
                "requested_model_revision": MODEL_REV,
                "resolved_model_revision": (
                    resolved_revision
                ),
                "sample_id": row["sample_id"],
                "utterance_id": row["utterance_id"],
                "prompt_id": pid,
                "prompt_level": spec["level"],
                "prompt_family": spec.get(
                    "family", ""
                ),
                "prompt_hash": spec["prompt_hash"],
                "prompt_text": spec["text"],
                "shot_count": len(exemplars),
                "shot_sample_ids": [
                    x["sample_id"] for x in exemplars
                ],
                "audio_path": row["audio_path"],
                "speech_condition": (
                    row["speech_condition"]
                ),
                "reference_original": (
                    row["reference_original"]
                ),
                "reference_dataset_normalized": (
                    row[
                        "reference_dataset_normalized"
                    ]
                ),
                "reference_normalized": (
                    row["reference_normalized"]
                ),
                "status": "success",
                "raw_output": "",
                "cleaned_prediction": "",
                "prediction_normalized": "",
                "generated_tokens": 0,
                "inference_seconds": 0.0,
                "error_type": "",
                "error_message": "",
                "timestamp": datetime.now().isoformat(),
                **known_metadata_from_row(row),
            }

            try:
                t0 = time.time()
                raw, n_new = transcribe_one(
                    row["audio_path"],
                    spec["text"],
                    exemplars,
                )
                record["inference_seconds"] = round(
                    time.time() - t0,
                    3,
                )
                record["generated_tokens"] = n_new
                record["raw_output"] = raw
                record["cleaned_prediction"] = (
                    clean_output(raw)
                )
                record["prediction_normalized"] = (
                    normalize_for_eval(
                        record["cleaned_prediction"]
                    )
                )

                if not record["cleaned_prediction"]:
                    record["status"] = "empty_output"

            except torch.cuda.OutOfMemoryError as e:
                record["status"] = "out_of_memory"
                record["error_type"] = type(e).__name__
                record["error_message"] = str(e)
                gc.collect()
                torch.cuda.empty_cache()

            except Exception as e:
                record["status"] = "inference_error"
                record["error_type"] = type(e).__name__
                record["error_message"] = str(e)

            with jsonlines.open(
                output_path,
                mode="a",
            ) as writer:
                writer.write(record)

            completed.add(key)
            pbar.update(1)

    pbar.close()
    print("Saved:", output_path)
    return output_path


## H. Shared scoring and diagnostics


In [ ]:
from jiwer import process_words, process_characters
from collections import Counter

DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
LATIN_RE = re.compile(r"[A-Za-z]")

REJECTION_PATTERNS = [
    r"\bi can(?:not|'t)\b",
    r"\bunable to\b",
    r"\bsorry\b",
    r"\bno audio\b",
    r"\bcannot transcribe\b",
    r"माफ\s*गर्नु",
    r"सुन्न\s*सक",
]

META_PATTERNS = [
    r"\bhere is\b.*\btranscri",
    r"\bthe transcription\b",
    r"\btranscription is\b",
    r"\btranscript is\b",
    r"\baudio (?:says|contains)\b",
]

def repetition_flag(text):
    tokens = normalize_for_eval(text).split()
    if len(tokens) < 6:
        return False

    run = 1
    for i in range(1, len(tokens)):
        if tokens[i] == tokens[i - 1]:
            run += 1
            if run >= 5:
                return True
        else:
            run = 1

    trigrams = [
        tuple(tokens[i:i + 3])
        for i in range(len(tokens) - 2)
    ]
    return bool(
        trigrams and max(Counter(trigrams).values()) >= 3
    )

def metric_record(ref, hyp):
    if not ref:
        return {"metric_status": "missing_reference"}

    try:
        w = process_words(ref, hyp)
        c = process_characters(ref, hyp)

        ref_words = w.hits + w.substitutions + w.deletions
        ref_chars = c.hits + c.substitutions + c.deletions

        return {
            "metric_status": "ok",
            "wer": float(w.wer),
            "cer": float(c.cer),
            "word_hits": int(w.hits),
            "word_substitutions": int(w.substitutions),
            "word_deletions": int(w.deletions),
            "word_insertions": int(w.insertions),
            "ref_words": int(ref_words),
            "word_edits": int(
                w.substitutions + w.deletions + w.insertions
            ),
            "char_hits": int(c.hits),
            "char_substitutions": int(c.substitutions),
            "char_deletions": int(c.deletions),
            "char_insertions": int(c.insertions),
            "ref_chars": int(ref_chars),
            "char_edits": int(
                c.substitutions + c.deletions + c.insertions
            ),
        }
    except Exception as e:
        return {
            "metric_status": "metric_error",
            "metric_error_type": type(e).__name__,
            "metric_error_message": str(e),
        }

def behavior_record(ref, hyp, metric):
    ref_words = max(
        int(metric.get("ref_words", len(ref.split())) or 0),
        1,
    )
    hyp_words = len(hyp.split())
    length_ratio = hyp_words / ref_words

    insertion_rate = (
        float(metric.get("word_insertions", 0)) / ref_words
        if metric.get("metric_status") == "ok"
        else np.nan
    )

    lower = hyp.lower()
    rejection = any(
        re.search(p, lower, flags=re.IGNORECASE)
        for p in REJECTION_PATTERNS
    )
    meta_response = any(
        re.search(p, lower, flags=re.IGNORECASE)
        for p in META_PATTERNS
    )
    repeated = repetition_flag(hyp)

    reasons = []
    if length_ratio > 2.5:
        reasons.append("length_ratio>2.5")
    if pd.notna(insertion_rate) and insertion_rate > 1.0:
        reasons.append("insertion_rate>1.0")
    if meta_response:
        reasons.append("meta_response")
    if repeated:
        reasons.append("repetition")

    ref_has_dev = bool(DEVANAGARI_RE.search(ref))
    ref_has_lat = bool(LATIN_RE.search(ref))
    hyp_has_dev = bool(DEVANAGARI_RE.search(hyp))
    hyp_has_lat = bool(LATIN_RE.search(hyp))

    script_presence_match = (
        (not ref_has_dev or hyp_has_dev)
        and (not ref_has_lat or hyp_has_lat)
    )

    return {
        "hyp_words": hyp_words,
        "output_reference_word_ratio": round(
            length_ratio, 4
        ),
        "insertion_rate": (
            round(insertion_rate, 4)
            if pd.notna(insertion_rate)
            else np.nan
        ),
        "rejection_flag": rejection,
        "meta_response_flag": meta_response,
        "repetition_flag": repeated,
        "hallucination_proxy_flag": bool(reasons),
        "hallucination_proxy_reason": "|".join(reasons),
        "reference_has_devanagari": ref_has_dev,
        "reference_has_latin": ref_has_lat,
        "hypothesis_has_devanagari": hyp_has_dev,
        "hypothesis_has_latin": hyp_has_lat,
        "script_presence_match": script_presence_match,
    }

def score_jsonl(jsonl_path, prompt_specs):
    valid_hashes = {
        spec["prompt_hash"]
        for spec in prompt_specs.values()
    }

    rows = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            if obj.get("prompt_hash") not in valid_hashes:
                continue
            if str(obj.get("resolved_model_revision", "")) != str(
                resolved_revision
            ):
                continue

            if obj["status"] in {"success", "empty_output"}:
                m = metric_record(
                    obj.get("reference_normalized", ""),
                    obj.get("prediction_normalized", ""),
                )
                obj.update(m)
                obj.update(
                    behavior_record(
                        obj.get("reference_normalized", ""),
                        obj.get("prediction_normalized", ""),
                        m,
                    )
                )
            else:
                obj["metric_status"] = (
                    "not_scored_inference_failure"
                )

            rows.append(obj)

    return pd.DataFrame(rows)

def summarize_group(g):
    metric_g = g[g["metric_status"] == "ok"].copy()

    ref_words = metric_g["ref_words"].sum()
    word_edits = metric_g["word_edits"].sum()
    ref_chars = metric_g["ref_chars"].sum()
    char_edits = metric_g["char_edits"].sum()

    return {
        "n_expected": len(g),
        "n_scored": len(metric_g),
        "coverage": (
            len(metric_g) / len(g) if len(g) else np.nan
        ),
        "failure_rate": (
            ~g["status"].isin(["success", "empty_output"])
        ).mean(),
        "empty_output_rate": (
            g["status"] == "empty_output"
        ).mean(),
        "corpus_wer": (
            word_edits / ref_words if ref_words else np.nan
        ),
        "corpus_cer": (
            char_edits / ref_chars if ref_chars else np.nan
        ),
        "macro_utterance_wer": metric_g["wer"].mean(),
        "median_utterance_wer": metric_g["wer"].median(),
        "macro_utterance_cer": metric_g["cer"].mean(),
        "hallucination_proxy_rate": (
            metric_g["hallucination_proxy_flag"].mean()
        ),
        "rejection_rate": (
            metric_g["rejection_flag"].mean()
        ),
        "meta_response_rate": (
            metric_g["meta_response_flag"].mean()
        ),
        "script_presence_match_rate": (
            metric_g["script_presence_match"].mean()
        ),
        "mean_output_reference_word_ratio": (
            metric_g["output_reference_word_ratio"].mean()
        ),
        "mean_inference_seconds": (
            g["inference_seconds"].replace(0, np.nan).mean()
        ),
        "total_ref_words": int(ref_words),
        "total_word_edits": int(word_edits),
    }

def grouped_summary(df, keys):
    rows = []
    grouped = df.groupby(
        keys,
        dropna=False,
        observed=False,
    )

    for group_key, group_df in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = dict(zip(keys, group_key))
        row.update(summarize_group(group_df))
        rows.append(row)

    return pd.DataFrame(rows)

SELECTION_SORT = [
    "failure_rate",
    "corpus_wer",
    "corpus_cer",
    "hallucination_proxy_rate",
]

def score_and_summarize_level(
    level,
    jsonl_path,
    prompt_specs,
):
    df = score_jsonl(jsonl_path, prompt_specs)

    expected = len(prompt_specs)
    expected *= (
        len(fewshot_eval_manifest)
        if level in {"L3", "L4"}
        else len(manifest)
    )
    if len(df) != expected:
        raise RuntimeError(
            f"{level}: expected {expected} scored records, "
            f"found {len(df)}."
        )

    df.to_csv(
        RUN_DIR / f"{level}_utterance_metrics.csv",
        index=False,
    )

    summary = grouped_summary(
        df,
        ["prompt_level", "prompt_id", "prompt_hash"],
    ).sort_values(SELECTION_SORT)

    summary.to_csv(
        RUN_DIR / f"{level}_prompt_summary.csv",
        index=False,
    )

    condition_summary = grouped_summary(
        df,
        ["prompt_id", "prompt_hash", "speech_condition"],
    )
    condition_summary.to_csv(
        RUN_DIR / f"{level}_condition_summary.csv",
        index=False,
    )

    print("Prompt summary:")
    display(summary)

    print("By speech condition:")
    display(condition_summary[[
        "prompt_id", "speech_condition", "n_scored",
        "corpus_wer", "corpus_cer",
        "hallucination_proxy_rate",
        "empty_output_rate", "failure_rate",
    ]])

    noisy = df[
        df["speech_condition"] == "noisy"
    ].copy()
    if len(noisy):
        noise_keys = ["prompt_id", "noise_type"]
        if "snr_db" in noisy.columns:
            noise_keys.append("snr_db")

        noise_summary = grouped_summary(
            noisy, noise_keys
        )
        noise_summary.to_csv(
            RUN_DIR / f"{level}_noise_summary.csv",
            index=False,
        )

    cs = df[
        df["speech_condition"] == "codeswitched"
    ].copy()
    if len(cs):
        if (
            "cmi_band" not in cs.columns
            or cs["cmi_band"].isna().all()
        ):
            cs["cmi_numeric"] = pd.to_numeric(
                cs.get("cmi", np.nan),
                errors="coerce",
            )
            cs["cmi_band_eval"] = pd.cut(
                cs["cmi_numeric"],
                bins=[-np.inf, 10, 20, np.inf],
                labels=["<=10", "10-20", ">20"],
            )
            cmi_key = "cmi_band_eval"
        else:
            cmi_key = "cmi_band"

        cmi_summary = grouped_summary(
            cs,
            ["prompt_id", cmi_key],
        )
        cmi_summary.to_csv(
            RUN_DIR / f"{level}_cmi_summary.csv",
            index=False,
        )

    return df, summary, condition_summary

def paired_bootstrap_winner_vs_challenger(
    df,
    winner_id,
    challenger_id,
    B=2000,
    seed=42,
):
    cols = ["sample_id", "word_edits", "ref_words"]

    a = df[
        (df["prompt_id"] == winner_id)
        & (df["metric_status"] == "ok")
    ][cols].copy()

    b = df[
        (df["prompt_id"] == challenger_id)
        & (df["metric_status"] == "ok")
    ][cols].copy()

    merged = a.merge(
        b,
        on="sample_id",
        suffixes=("_winner", "_challenger"),
    )
    if len(merged) < 2:
        return None

    rng = np.random.default_rng(seed)
    diffs = np.empty(B, dtype=float)
    n = len(merged)

    for i in range(B):
        idx = rng.integers(0, n, size=n)
        sample = merged.iloc[idx]

        winner_den = sample[
            "ref_words_winner"
        ].sum()
        challenger_den = sample[
            "ref_words_challenger"
        ].sum()

        if winner_den == 0 or challenger_den == 0:
            diffs[i] = np.nan
            continue

        winner_wer = (
            sample["word_edits_winner"].sum()
            / winner_den
        )
        challenger_wer = (
            sample["word_edits_challenger"].sum()
            / challenger_den
        )
        diffs[i] = winner_wer - challenger_wer

    diffs = diffs[~np.isnan(diffs)]
    if not len(diffs):
        return None

    lo, hi = np.quantile(diffs, [0.025, 0.975])

    return {
        "winner": winner_id,
        "challenger": challenger_id,
        "n_paired": n,
        "mean_delta_wer": float(diffs.mean()),
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "winner_better_probability": float(
            np.mean(diffs < 0)
        ),
        "stable_direction_95pct": bool(hi < 0),
    }

def show_level_selection(df, summary):
    winner = summary.sort_values(
        SELECTION_SORT
    ).iloc[0]["prompt_id"]

    print("Selected by development rule:", winner)

    challengers = [
        pid
        for pid in summary["prompt_id"].unique()
        if pid != winner
    ]

    rows = []
    for challenger in challengers:
        result = paired_bootstrap_winner_vs_challenger(
            df,
            winner,
            challenger,
            B=2000,
            seed=SEED,
        )
        if result:
            rows.append(result)

    bootstrap_df = pd.DataFrame(rows)
    if len(bootstrap_df):
        display(bootstrap_df)

    return winner, bootstrap_df


## I. Sanity check


In [ ]:
sanity_row = manifest.iloc[0]
sanity_raw, sanity_new_tokens = transcribe_one(
    sanity_row["audio_path"],
    PROMPT_TEXTS["L0_a"],
    exemplars=None,
)

print("Sample:", sanity_row["sample_id"])
print("Reference:", sanity_row["reference_normalized"])
print("Prediction:", clean_output(sanity_raw))
print("Generated tokens:", sanity_new_tokens)


## J. L0 — Minimal control

Run the fixed minimal control on all 30 development utterances.


In [ ]:
L0_SPECS = make_specs(['L0_a'])
L0_PATH = RUN_DIR / "L0_predictions.jsonl"

run_pipeline(
    manifest_df=manifest,
    prompt_specs=L0_SPECS,
    output_path=L0_PATH,
)


In [ ]:
L0_DF, L0_PROMPT_SUMMARY, L0_CONDITION_SUMMARY = score_and_summarize_level(
    level="L0",
    jsonl_path=L0_PATH,
    prompt_specs=L0_SPECS,
)

L0_SELECTED = "L0_a"
L0_BOOTSTRAP = pd.DataFrame()
print("Selected L0 prompt:", L0_SELECTED)


## K. L1 — Task-instruction selection

Evaluate all L1 candidates on the same 30 utterances and select by the predeclared rule.


In [ ]:
L1_SPECS = make_specs(['L1_a', 'L1_b', 'L1_c'])
L1_PATH = RUN_DIR / "L1_predictions.jsonl"

run_pipeline(
    manifest_df=manifest,
    prompt_specs=L1_SPECS,
    output_path=L1_PATH,
)


In [ ]:
L1_DF, L1_PROMPT_SUMMARY, L1_CONDITION_SUMMARY = score_and_summarize_level(
    level="L1",
    jsonl_path=L1_PATH,
    prompt_specs=L1_SPECS,
)

L1_SELECTED, L1_BOOTSTRAP = show_level_selection(
    L1_DF,
    L1_PROMPT_SUMMARY,
)
L1_BOOTSTRAP.to_csv(
    RUN_DIR / "L1_prompt_bootstrap.csv",
    index=False,
)
print("Selected L1 prompt:", L1_SELECTED)


## L. L2 — Linguistically aware selection

Evaluate all L2 candidates on the same 30 utterances and select by the same rule.


In [ ]:
L2_SPECS = make_specs(['L2_a', 'L2_b', 'L2_c'])
L2_PATH = RUN_DIR / "L2_predictions.jsonl"

run_pipeline(
    manifest_df=manifest,
    prompt_specs=L2_SPECS,
    output_path=L2_PATH,
)


In [ ]:
L2_DF, L2_PROMPT_SUMMARY, L2_CONDITION_SUMMARY = score_and_summarize_level(
    level="L2",
    jsonl_path=L2_PATH,
    prompt_specs=L2_SPECS,
)

L2_SELECTED, L2_BOOTSTRAP = show_level_selection(
    L2_DF,
    L2_PROMPT_SUMMARY,
)
L2_BOOTSTRAP.to_csv(
    RUN_DIR / "L2_prompt_bootstrap.csv",
    index=False,
)
print("Selected L2 prompt:", L2_SELECTED)


## M. L3 — One-shot prompting

Use the selected L2 prompt with the fixed clean exemplar. Evaluate on the 27 non-exemplar targets.


In [ ]:
L3_BASE_PROMPT_ID = L2_SELECTED
L3_EXEMPLARS = [SHOT_BANK["clean"]]

L3_SPECS = {
    "L3_a": {
        "prompt_id": "L3_a",
        "level": "L3",
        "family": "1shot_from_" + L3_BASE_PROMPT_ID,
        "text": PROMPT_TEXTS[L3_BASE_PROMPT_ID],
        "shot_count": 1,
        "prompt_hash": prompt_hash(
            PROMPT_TEXTS[L3_BASE_PROMPT_ID],
            [
                {
                    "sample_id": x["sample_id"],
                    "transcript": x["demonstration_transcript"],
                }
                for x in L3_EXEMPLARS
            ],
        ),
    }
}

L3_PATH = RUN_DIR / "L3_predictions.jsonl"

run_pipeline(
    manifest_df=fewshot_eval_manifest,
    prompt_specs=L3_SPECS,
    output_path=L3_PATH,
    exemplar_map={"L3_a": L3_EXEMPLARS},
)


In [ ]:
L3_DF, L3_PROMPT_SUMMARY, L3_CONDITION_SUMMARY = score_and_summarize_level(
    level="L3",
    jsonl_path=L3_PATH,
    prompt_specs=L3_SPECS,
)

L3_SELECTED = "L3_a"
print("Selected L3 prompt:", L3_SELECTED)
print("Base prompt:", L3_BASE_PROMPT_ID)


## N. L4 — Three-shot prompting

Use the selected L2 prompt with fixed clean, code-switched, and noisy exemplars. Evaluate on the same 27 targets.


In [ ]:
L4_BASE_PROMPT_ID = L2_SELECTED
L4_EXEMPLARS = [SHOT_BANK["clean"], SHOT_BANK["codeswitched"], SHOT_BANK["noisy"]]

L4_SPECS = {
    "L4_a": {
        "prompt_id": "L4_a",
        "level": "L4",
        "family": "3shot_from_" + L4_BASE_PROMPT_ID,
        "text": PROMPT_TEXTS[L4_BASE_PROMPT_ID],
        "shot_count": 3,
        "prompt_hash": prompt_hash(
            PROMPT_TEXTS[L4_BASE_PROMPT_ID],
            [
                {
                    "sample_id": x["sample_id"],
                    "transcript": x["demonstration_transcript"],
                }
                for x in L4_EXEMPLARS
            ],
        ),
    }
}

L4_PATH = RUN_DIR / "L4_predictions.jsonl"

run_pipeline(
    manifest_df=fewshot_eval_manifest,
    prompt_specs=L4_SPECS,
    output_path=L4_PATH,
    exemplar_map={"L4_a": L4_EXEMPLARS},
)


In [ ]:
L4_DF, L4_PROMPT_SUMMARY, L4_CONDITION_SUMMARY = score_and_summarize_level(
    level="L4",
    jsonl_path=L4_PATH,
    prompt_specs=L4_SPECS,
)

L4_SELECTED = "L4_a"
print("Selected L4 prompt:", L4_SELECTED)
print("Base prompt:", L4_BASE_PROMPT_ID)


## O. Cross-level comparison

Compare the selected L0-L4 configurations on the same 27 non-exemplar targets.


In [ ]:
common_ids = set(fewshot_eval_manifest["sample_id"])

selected_zero_shot = pd.concat(
    [
        L0_DF[L0_DF["prompt_id"] == L0_SELECTED],
        L1_DF[L1_DF["prompt_id"] == L1_SELECTED],
        L2_DF[L2_DF["prompt_id"] == L2_SELECTED],
    ],
    ignore_index=True,
)
selected_zero_shot = selected_zero_shot[
    selected_zero_shot["sample_id"].isin(common_ids)
].copy()

selected_all = pd.concat(
    [selected_zero_shot, L3_DF, L4_DF],
    ignore_index=True,
)

expected_rows = len(fewshot_eval_manifest) * 5
assert len(selected_all) == expected_rows, (
    f"Expected {expected_rows} common-target rows, found {len(selected_all)}."
)

cross_level_summary = grouped_summary(
    selected_all,
    ["prompt_level", "prompt_id"],
).sort_values(["prompt_level"])

cross_level_condition = grouped_summary(
    selected_all,
    ["prompt_level", "speech_condition"],
).sort_values(["prompt_level", "speech_condition"])

cross_level_summary.to_csv(
    RUN_DIR / "cross_level_common_27_summary.csv",
    index=False,
)
cross_level_condition.to_csv(
    RUN_DIR / "cross_level_common_27_condition_summary.csv",
    index=False,
)

display(
    cross_level_summary[
        [
            "prompt_level",
            "prompt_id",
            "n_scored",
            "failure_rate",
            "corpus_wer",
            "corpus_cer",
            "macro_utterance_wer",
            "hallucination_proxy_rate",
            "rejection_rate",
            "script_presence_match_rate",
            "mean_inference_seconds",
        ]
    ]
)


## P. Freeze the prompt-development result

Save the selected prompts, hashes, demonstrations, manifest identity, and model revision for final benchmarking.


In [ ]:
selected_level_specs = {
    "L0": L0_SPECS[L0_SELECTED],
    "L1": L1_SPECS[L1_SELECTED],
    "L2": L2_SPECS[L2_SELECTED],
    "L3": L3_SPECS[L3_SELECTED],
    "L4": L4_SPECS[L4_SELECTED],
}

frozen_bundle = {
    "bundle_version": 1,
    "created_at_utc": datetime.utcnow().replace(
        microsecond=0
    ).isoformat() + "Z",
    "model_id": MODEL_ID,
    "requested_model_revision": MODEL_REV,
    "resolved_model_revision": resolved_revision,
    "precision": PRECISION,
    "quantization": QUANTIZATION,
    "manifest_path": str(PROMPT_DEV_MANIFEST_PATH),
    "manifest_hash": MANIFEST_HASH,
    "manifest_n": len(manifest),
    "promptset_hash": PROMPTSET_HASH,
    "selection_rule": {
        "L0": "Fixed minimal control.",
        "L1": (
            "Lowest failure rate, corpus WER, corpus CER, "
            "then hallucination-proxy rate."
        ),
        "L2": (
            "Lowest failure rate, corpus WER, corpus CER, "
            "then hallucination-proxy rate."
        ),
        "L3": (
            "Selected L2 prompt with one fixed clean "
            "development exemplar."
        ),
        "L4": (
            "Selected L2 prompt with fixed clean, code-switched, "
            "and noisy development exemplars."
        ),
        "bootstrap_role": "Diagnostic stability analysis only.",
    },
    "normalization_rules": NORMALIZATION_RULES,
    "selected_levels": {
        level: {
            "prompt_id": spec["prompt_id"],
            "prompt_family": spec["family"],
            "prompt_text": spec["text"],
            "prompt_hash": spec["prompt_hash"],
            "shot_count": spec["shot_count"],
            "shot_ids": (
                []
                if level in {"L0", "L1", "L2"}
                else (
                    [x["sample_id"] for x in L3_EXEMPLARS]
                    if level == "L3"
                    else [x["sample_id"] for x in L4_EXEMPLARS]
                )
            ),
        }
        for level, spec in selected_level_specs.items()
    },
    "fewshot_exemplars": {
        key: {
            "sample_id": value["sample_id"],
            "speech_condition": value["speech_condition"],
            "duration_sec": value["duration_sec"],
            "transcript": value["demonstration_transcript"],
        }
        for key, value in SHOT_BANK.items()
    },
    "evaluation_note": (
        "Prompt development only. Few-shot exemplars are excluded "
        "from all cross-level comparisons. Final benchmark data "
        "must remain unseen."
    ),
    "hallucination_note": (
        "Hallucination proxy is a heuristic diagnostic based on "
        "excessive length, insertions, meta-response behavior, "
        "or repetition."
    ),
}

bundle_path = RUN_DIR / "frozen_prompt_bundle.json"
bundle_path.write_text(
    json.dumps(frozen_bundle, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

used_ids = manifest[
    ["sample_id", "speech_condition", "file", "audio_path"]
].copy()
used_ids["used_as_fewshot_exemplar"] = (
    used_ids["sample_id"].isin(shot_ids_all)
)
used_ids.to_csv(
    RUN_DIR / "prompt_dev_used_ids.csv",
    index=False,
)

selected_prompt_summary = pd.DataFrame(
    [
        {
            "prompt_level": level,
            "prompt_id": spec["prompt_id"],
            "prompt_family": spec["family"],
            "prompt_hash": spec["prompt_hash"],
            "shot_count": spec["shot_count"],
        }
        for level, spec in selected_level_specs.items()
    ]
)
selected_prompt_summary.to_csv(
    RUN_DIR / "selected_prompt_summary.csv",
    index=False,
)

print("Frozen prompt bundle:", bundle_path)
display(selected_prompt_summary)


## Q. Benchmark-readiness check

Do not begin the final benchmark until every check passes.


In [ ]:
checks = []

def add_check(name, passed, detail):
    checks.append(
        {
            "check": name,
            "passed": bool(passed),
            "detail": str(detail),
        }
    )

add_check(
    "development_manifest_size",
    len(manifest) == EXPECTED_DEV_N,
    f"{len(manifest)}/{EXPECTED_DEV_N}",
)
add_check(
    "audio_files_resolved",
    manifest["audio_path"].map(
        lambda x: Path(x).exists()
    ).all(),
    (
        f"{manifest['audio_path'].map(lambda x: Path(x).exists()).sum()}"
        f"/{len(manifest)}"
    ),
)
add_check(
    "references_nonempty",
    manifest["reference_normalized"].str.len().gt(0).all(),
    (
        f"{manifest['reference_normalized'].str.len().gt(0).sum()}"
        f"/{len(manifest)}"
    ),
)
add_check(
    "L0_record_count",
    len(L0_DF) == len(manifest),
    len(L0_DF),
)
add_check(
    "L1_record_count",
    len(L1_DF) == len(manifest) * len(L1_SPECS),
    len(L1_DF),
)
add_check(
    "L2_record_count",
    len(L2_DF) == len(manifest) * len(L2_SPECS),
    len(L2_DF),
)
add_check(
    "L3_record_count",
    len(L3_DF) == len(fewshot_eval_manifest),
    len(L3_DF),
)
add_check(
    "L4_record_count",
    len(L4_DF) == len(fewshot_eval_manifest),
    len(L4_DF),
)

all_scored = pd.concat(
    [L0_DF, L1_DF, L2_DF, L3_DF, L4_DF],
    ignore_index=True,
)
failure_count = int(
    (~all_scored["status"].isin(["success", "empty_output"])).sum()
)
metric_error_count = int(
    (all_scored["metric_status"] != "ok").sum()
)

add_check(
    "no_inference_failures",
    failure_count == 0,
    failure_count,
)
add_check(
    "no_metric_failures",
    metric_error_count == 0,
    metric_error_count,
)
add_check(
    "fewshot_exemplars_excluded",
    shot_ids_all.isdisjoint(
        set(fewshot_eval_manifest["sample_id"])
    ),
    len(shot_ids_all),
)
add_check(
    "resolved_revision_recorded",
    bool(resolved_revision),
    resolved_revision,
)
add_check(
    "frozen_bundle_written",
    (RUN_DIR / "frozen_prompt_bundle.json").exists(),
    RUN_DIR / "frozen_prompt_bundle.json",
)

readiness = pd.DataFrame(checks)
readiness.to_csv(
    RUN_DIR / "benchmark_readiness.csv",
    index=False,
)
display(readiness)

ready = bool(readiness["passed"].all())
print(
    "READY FOR FINAL BENCHMARK:",
    "YES" if ready else "NO",
)
if not ready:
    print("Resolve failed checks before the final benchmark.")


## R. Archive prompt-development outputs


In [ ]:
import shutil

archive_base = (
    Path("/kaggle/working")
    / f"{RESULTS_ROOT.name}_prompt_development_results"
)
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=RUN_DIR,
)
print("Results archive:", archive_path)


## Research reporting notes

- Keep the final benchmark isolated from prompt development.
- Freeze prompt text, few-shot examples, model revision, decoding, and normalization before benchmarking.
- Report corpus WER as the primary accuracy metric and corpus CER as the main secondary accuracy metric.
- Report failure, empty-output, hallucination-proxy, rejection, script-match, and inference-time diagnostics.
- Report condition-level results for clean, noisy, and code-switched speech.
- Treat paired bootstrap intervals as prompt-selection stability diagnostics, not as the selection rule.
- Compare L0-L4 on the same 27 non-exemplar development targets.
- Treat the hallucination proxy as a heuristic, not semantic hallucination ground truth.
